# 00_env_config

Environment bootstrap for FabricOps Starter Kit notebooks.
This notebook defines environment-wide values and assembles framework config.
Reusable functions come from `fabricops_kit` package modules.


In [ ]:
from fabricops_kit.fabric_input_output import (
    FabricStore,
    check_naming_convention,
    read_lakehouse_csv,
    read_lakehouse_table,
    write_lakehouse_table,
    read_warehouse_table,
    write_warehouse_table,
)
from fabricops_kit.config import (
    AIPromptConfig,
    FrameworkConfig,
    GovernanceConfig,
    LineageConfig,
    NotebookRuntimeConfig,
    PathConfig,
    QualityConfig,
    ReviewWorkflowConfig,
    load_config,
    setup_notebook,
)


In [ ]:
ENV = "dev"
VALIDATION_MODE = "warn"
NOTEBOOK_PREFIXES = ("00_env_config", "01_dsa_", "02_ex_", "03_pc_")


In [ ]:
ENV_PATHS = {
    "dev": {
        "source": FabricStore(env="dev", workspace_id="00000000-0000-0000-0000-000000000001", item_id="11111111-1111-1111-1111-111111111111", name="lh_source_dev", kind="lakehouse"),
        "unified": FabricStore(env="dev", workspace_id="00000000-0000-0000-0000-000000000001", item_id="22222222-2222-2222-2222-222222222222", name="lh_unified_dev", kind="lakehouse"),
        "product": FabricStore(env="dev", workspace_id="00000000-0000-0000-0000-000000000001", item_id="33333333-3333-3333-3333-333333333333", name="wh_product_dev", kind="warehouse"),
        "metadata": FabricStore(env="dev", workspace_id="00000000-0000-0000-0000-000000000001", item_id="44444444-4444-4444-4444-444444444444", name="lh_metadata_dev", kind="lakehouse"),
    }
}


## AI prompt overrides


### Business context prompt


In [ ]:
# Prompt text lives in 00_env_config for visibility and environment-level ownership.
BUSINESS_CONTEXT_PROMPT_TEMPLATE = """
Infer business meaning only for one column. Do not classify personal data.
Use table_name={table_name}, table_context={table_context}, column_name={column_name}, data_type={data_type},
row_count={row_count}, null_count={null_count}, distinct_count={distinct_count}, observed_values_sample={observed_values_sample}.
Return only Python dict:
BUSINESS_CONTEXT = {"column_name": "name", "business_context": "clear business meaning", "notes": "optional reviewer note"}
""".strip()


### DQ rule suggestion prompt


In [ ]:
DQ_RULE_SUGGESTION_PROMPT_TEMPLATE = """
You are helping draft candidate FabricOps data quality rules for a pipeline contract notebook.

These suggestions are advisory only.
A human engineer, data steward, or governance reviewer must approve them before enforcement.

Use only these FabricOps rule_type values:

1. not_null
   Use when a column must be populated.
   Required fields:
   rule_id, rule_type, columns, severity, description

2. unique_key
   Use when one or more columns define the business grain and must be unique.
   Required fields:
   rule_id, rule_type, columns, severity, description

3. accepted_values
   Use when a column should only contain known business values.
   Required fields:
   rule_id, rule_type, columns, allowed_values, severity, description

4. value_range
   Use when a numeric, date, or timestamp column should stay within a sensible range.
   Required fields:
   rule_id, rule_type, columns, lower_bound or upper_bound, severity, description

5. regex_format
   Use when a string column should match a known format such as email, code, phone, postal code, or ID.
   Required fields:
   rule_id, rule_type, columns, regex_pattern, severity, description

Heuristics:
- Suggest not_null when null_count is 0 or when the column name looks mandatory, such as id, key, date, code, status, amount, or name.
- Suggest unique_key only when distinct_count is close to row_count and the column name looks like an identifier or business key.
- Suggest accepted_values when distinct_count is small and the observed values look like business categories.
- Suggest value_range only when lower_bound and upper_bound are available and the range is business meaningful.
- Suggest regex_format only for clear format columns such as email, phone, postal_code, programme_code, course_code, invoice_number, or staff_id.
- Use severity="error" only for rules that should block the pipeline.
- Use severity="warning" for rules that should be reviewed but should not block the pipeline.
- Do not suggest unsupported rule types.
- Do not return Great Expectations, Deequ, DQX, SQL, or pseudocode syntax.

Return only a Python dictionary named DQ_RULES using this shape:

DQ_RULES = {
    "{table_name}": [
        {
            "rule_id": "lower_snake_case_rule_id",
            "rule_type": "one_supported_rule_type",
            "columns": ["column_name"],
            "severity": "error_or_warning",
            "description": "Plain business explanation."
        }
    ]
}

For accepted_values, include allowed_values.
For value_range, include lower_bound and/or upper_bound.
For regex_format, include regex_pattern.

Table name:
{table_name}

Column profile row:
Column name: {column_name}
Data type: {data_type}
Row count: {row_count}
Null count: {null_count}
Null percent: {null_percent}
Distinct count: {distinct_count}
Distinct percent: {distinct_percent}
Minimum value: {min_value}
Maximum value: {max_value}
Observed values sample: {observed_values_sample}

Approved business context:
{approved_business_context}
""".strip()


### Governance personal identifier prompt


In [ ]:
GOVERNANCE_PERSONAL_IDENTIFIER_PROMPT_TEMPLATE = """
Use approved_business_context as evidence. Classify personal identifier status separately from business context.
Allowed personal identifier values: not_personal_data, direct_identifier, indirect_identifier, unknown.
Allowed confidentiality labels: public, confidential, restricted.
Return only JSON dict with keys:
column_name, ai_suggested_personal_identifier_classification, confidentiality_label, evidence, reasoning.
""".strip()


### Governance candidate classification prompt


In [ ]:
GOVERNANCE_CANDIDATE_PROMPT_TEMPLATE = (
    "Generate governance label suggestions from profile metadata. "
    "Return JSON only with: table_name, column_name, candidate_label, reason, evidence, needs_human_review. "
    "Allowed candidate_label: public, internal, confidential_candidate, restricted_candidate, unknown. "
    "Suggestions are for human review and are not deterministic enforcement. "
    "Dataset name: {dataset_name}. Business context: {business_context}. "
    "Row profile fields: table_name={table_name}, column_name={column_name}, data_type={data_type}, profile_summary={profile_summary}."
)


### Governance review prompt


In [ ]:
GOVERNANCE_REVIEW_PROMPT_TEMPLATE = (
    "Use business_context={business_context}, approved_usage={approved_usage}, dataset_context={dataset_context}, "
    "profile_context={profile_context}, glossary_context={glossary_context}, steward_notes={steward_notes}. "
    "Return JSON rows with suggestion_type,target_column,approved_label,business_reason,evidence,confidence."
)


### Handover summary prompt


In [ ]:
HANDOVER_SUMMARY_PROMPT_TEMPLATE = (
    "Generate handover summary suggestions. "
    "Return JSON only with: pipeline_summary, important_transformations, business_reason, handover_notes, risks_or_open_questions. "
    "Suggestions are for human review and are not deterministic enforcement. "
    "Business context: {business_context}. "
    "Row summary field: summary={summary}."
)


In [ ]:
PATH_CONFIG = PathConfig(paths=ENV_PATHS)
RUNTIME_CONFIG = NotebookRuntimeConfig(
    allowed_notebook_prefixes=NOTEBOOK_PREFIXES,
)
AI_PROMPTS = AIPromptConfig(
    business_context_template=BUSINESS_CONTEXT_PROMPT_TEMPLATE,
    dq_rule_candidate_template=DQ_RULE_SUGGESTION_PROMPT_TEMPLATE,
    governance_personal_identifier_template=GOVERNANCE_PERSONAL_IDENTIFIER_PROMPT_TEMPLATE,
    governance_candidate_template=GOVERNANCE_CANDIDATE_PROMPT_TEMPLATE,
    governance_review_template=GOVERNANCE_REVIEW_PROMPT_TEMPLATE,
    handover_summary_template=HANDOVER_SUMMARY_PROMPT_TEMPLATE,
)
QUALITY_CONFIG = QualityConfig()
GOVERNANCE_CONFIG = GovernanceConfig()
REVIEW_WORKFLOW_CONFIG = ReviewWorkflowConfig()
LINEAGE_CONFIG = LineageConfig()

CONFIG = FrameworkConfig(
    path_config=PATH_CONFIG,
    notebook_runtime_config=RUNTIME_CONFIG,
    ai_prompt_config=AI_PROMPTS,
    quality_config=QUALITY_CONFIG,
    governance_config=GOVERNANCE_CONFIG,
    review_workflow_config=REVIEW_WORKFLOW_CONFIG,
    lineage_config=LINEAGE_CONFIG,
)


In [ ]:
CONFIG = load_config(CONFIG)
BOOTSTRAP = setup_notebook(
    config=CONFIG,
    env=ENV,
    required_targets=["source", "unified", "product", "metadata"],
    notebook_name="00_env_config",
)
check_naming_convention(
    allowed_prefixes=NOTEBOOK_PREFIXES,
    fail_on_error=(VALIDATION_MODE == "strict"),
)


In [ ]:
print("FabricOps environment bootstrap ready")
print(f"- env: {ENV}")
print(f"- validation mode: {VALIDATION_MODE}")
print(f"- source target: {CONFIG.path_config.paths[ENV]['source'].name}")
print(f"- metadata target: {CONFIG.path_config.paths[ENV]['metadata'].name}")
